# Carga del archivo e Importación de Librerías

In [2]:

import numpy as np 
import pandas as pd 
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest # Prueba de z de proporcines 
from statsmodels.stats.multitest import multipletests #para corrección bonferroni

df = pd.read_excel("diabetes.xlsx")
pd.set_option("display.max_rows", None)  # Mostrar todas las filas
pd.set_option("display.max_columns", None)  # Mostrar todas las columnas

# Punto 1

¿El  valor  promedio  de  colesterol  total  de  la  muestra,  correspondiente  a 
pacientes masculinos es menor 200mg/dL Y en las pacientes femeninas, es 
menor 200mg/dL?

Queremos comparar si la media de una muestra es diferente a un valor medio de referencia, tanto para hombres como para mujeres. Por lo tanto la _**prueba t para una sola muestra**_ nos puede servir.

- Variable: s1 $\rightarrow$ total serum cholesterol


- Hipótesis:
    - $H_0$: El valor  promedio  de  colesterol  total  de  la  muestra es igual o mayor a 200 mg/dL para pacientes másculinos y femeninos.
    - $H_1$: El valor  promedio  de  colesterol  total  de  la  muestra es menor a 200mg/dL para pacientes másculinos y femeninos.

La prueba t, supone que los datos tienen una distribución normal. Para probar la normalidad de los datos, hacemos una prueba de normalidad.

In [4]:
numero_masc = len(df[df['SEX'] == 2])

print(f"Total Masculino {numero_masc}")

numero_fem = len(df[df['SEX'] == 1])

print(f"Total Femenino {numero_fem}")

Total Masculino 207
Total Femenino 235


Como el total de datos es mayor a 50 en ambos casos, usamos la prueba de Kolmogorov-Smirnoff.

- Definición de Hipótesis para Kolmogorov-Smirnoff
    - $H_0$(Hipótesis Nula): Los datos sí siguen una distribución normal.
    - $H_1$(Hipótesis Alterna): Los datos no siguen una distribución normal.

In [5]:
alpha = 0.05

datos_hombres_s1 = df[df['SEX'] == 2]['S1']
ks_stat, ks_p = stats.kstest(datos_hombres_s1, 'norm', args=(np.mean(datos_hombres_s1), np.std(datos_hombres_s1)))

print("Para hombres:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S1 en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos no siguen una DISTRIBUCIÓN NORMAL.")

#------------------------------------------------------------

datos_mujeres_s1 = df[df['SEX'] == 1]['S1']
ks_stat, ks_p = stats.kstest(datos_hombres_s1, 'norm', args=(np.mean(datos_mujeres_s1), np.std(datos_mujeres_s1)))

print("Para mujeres:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S1 en mujeres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos no siguen una DISTRIBUCIÓN NORMAL.")

Para hombres:
El p-valor (0.2396770979328262) es mayor a 0.05. Asumimos que los datos de S1 en hombres son NORMALES.
Para mujeres:
El p-valor (0.761722343708202) es mayor a 0.05. Asumimos que los datos de S1 en mujeres son NORMALES.


Como en ambos casos, la distribución es normal, podemos efectuar la prueba t

In [6]:
mu = 200 # (mg/dL) El valor de referencia contra el cual comparamos 
alpha = 0.05

# Para Hombres
datos_hombres_s1 = df[df['SEX'] == 2]['S1']
t_stat, V_pMasc = stats.ttest_1samp(datos_hombres_s1, mu, alternative='less')

# Para Mujeres
datos_mujeres_s1 = df[df['SEX'] == 1]['S1'] # Datos de colesterol total para hombres
t_stat, V_pFem = stats.ttest_1samp(datos_mujeres_s1, mu, alternative='less')

if V_pMasc < alpha:
    print("Para Hombres H0 es Falso")
else:
    print("Para Hombres H0 es Verdadero")

if V_pFem < alpha:
    print("Para Mujeres H0 es Falso")
else:
    print("Para Mujeres H0 es Verdadero")


Para Hombres H0 es Falso
Para Mujeres H0 es Falso


Por lo tanto, tanto para hombres y mujeres, $H_1$ es verdadero; lo que significa que el valor  promedio  de  colesterol  total  de  la  muestra es menor a 200mg/dL para pacientes másculinos y femeninos.

# Punto 2

¿Hay diferencias entre el nivel de colesterol ldl ["S2"] entre pacientes con menos 
de 40 años y más de 40 años de edad?

Compararemos 2 grupos...
 
- Grupo 1: Pacientes con menos de 40 años.
- Grupo 2: Pacientes con más de 40 años.

Es decir, Evalúa si la diferencia de la media de dos muestras es significativamente diferentes de cero. Por lo tanto usaremos una _**Prueba T para dos muestras**_.

Determinamos la cantidad de pacientes mayores y menores a 40 años.

- Hipótesis:
    - $H_0$: La media del nivel de colesterol ldl entre pacientes con menos de 40 años y más de 40 años de edad son iguales.
    - $H_1$: La media del nivel de colesterol ldl entre pacientes con menos de 40 años y más de 40 años de edad son diferentes.

La _**Prueba T para dos muestras**_, tiene el supuesto de normalidad en las variables, homocedasticidad e independencia. De esta manera...

- Se asume el supuesto de independencia ya que los datos corresponden a individuos distintos y la medición de uno no influye en la del otro.

- Se asume normalidad en los datos. Así que hacemos una prueba de normalidad...

In [7]:
pacientes_mayores_40 = len(df[df['AGE'] > 40])
print(f"Total de pacientes con edad mayor a 40: {pacientes_mayores_40}")

pacientes_menores_40 = len(df[df['AGE'] < 40])
print(f"Total de pacientes con edad menor a 40: {pacientes_menores_40}")

Total de pacientes con edad mayor a 40: 320
Total de pacientes con edad menor a 40: 117


Como el total de datos es mayor a 50 en ambos casos, usamos la prueba de Kolmogorov-Smirnoff.

Hipótesis para Kolmogorov-Smirnoff
- $H_0$(Hipótesis Nula): Los datos sí siguen una distribución normal.
- $H_1$(Hipótesis Alterna): Los datos no siguen una distribución normal.

In [8]:
alpha = 0.05

pacientes_mayores_40 = df[df['AGE'] > 40]['S2']
ks_stat, ks_p = stats.kstest(pacientes_mayores_40, 'norm', args=(np.mean(pacientes_mayores_40), np.std(pacientes_mayores_40)))

print("Para Mayores de 40 años:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S2 en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de S2 no siguen una DISTRIBUCIÓN NORMAL.")

# ------------------------------------

pacientes_menores_40 = df[df['AGE'] < 40]['S2']
ks_stat, ks_p = stats.kstest(pacientes_menores_40, 'norm', args=(np.mean(pacientes_menores_40), np.std(pacientes_menores_40)))

print("Para Menores de 40 años:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S2 en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de S2 no siguen una DISTRIBUCIÓN NORMAL.")

Para Mayores de 40 años:
El p-valor (0.4272314401788271) es mayor a 0.05. Asumimos que los datos de S2 en hombres son NORMALES.
Para Menores de 40 años:
El p-valor (0.23024405303211082) es mayor a 0.05. Asumimos que los datos de S2 en hombres son NORMALES.


Ahora, hacemos prueba de _**Homocedasticidad**_, las hipotesis para esta prueba son:
- $H_0$: Los dos grupos de datos presentan varianza constante.
- $H_1$: Los dos grupos de datos no presentan varianza constante.

In [9]:
var1 = df[df['AGE'] > 40]['S2']
var2 = df[df['AGE'] < 40]['S2']
levene_stat, levene_p = stats.levene(var1, var2)

if ks_p > alpha:
    print(f"El p-valor ({levene_p}) es mayor a 0.05. Así que H0 es VERDADERO y los dos grupos de datos presentan varianza constante.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Así que H0 es FALSO y los dos grupos de datos no presentan varianza constante.")

El p-valor (0.7974128914697795) es mayor a 0.05. Así que H0 es VERDADERO y los dos grupos de datos presentan varianza constante.


Como ambos casos tienen una distribución normal y tienen una varianza constante (Homocedasticidad), podemos efectuar una _**Prueba T para dos muestras**_.

In [10]:
datos1 = df[df['AGE'] > 40]['S2']
datos2 = df[df['AGE'] < 40]['S2']
t_stat, p_value = stats.ttest_ind(datos1, datos2)

if p_value < alpha:
    print(f"El p-valor ({p_value}) es menor a 0.05. Por lo tanto H0 es FALSO.")
else:
    print(f"El p-valor ({p_value}) es mayor a 0.05. Por lo tanto H0 es VERDADERO.")

El p-valor (0.00030433241096305817) es menor a 0.05. Por lo tanto H0 es FALSO.


Por lo tanto, $H_1$ es verdadero; lo que indica que la media del nivel de colesterol ldl entre pacientes con menos de 40 años y más de 40 años de edad son diferentes.

# Punto 3

¿Hay  diferencia  en  la  medida  de  progresión  de  la  enfermedad  entre 
hombre y mujeres? ¿y entre pacientes con menos de 40 años y más de 40 
años de edad?

- Variable $\rightarrow$ Y, disease progression.

Haremos dos comparaciones:
 1. La media de progresión de la enfermedad entre hombres y mujeres.
 2. La media de progresión de la enfermedad entre pacientes con menos de 40 años y más de 40 
años de edad.

Es decir, para ambos casos evaluaremos si la diferencia de la media de dos muestras es significativamente diferentes de cero.

Para determinar que prueba usaremos, primero haremos una prueba de normalidad.

In [13]:
numero_masc = len(df[df['SEX'] == 2])
print(f"Total Masculino {numero_masc}")

numero_fem = len(df[df['SEX'] == 1])
print(f"Total Femenino {numero_fem}")

print("-------------------------------------------")

pacientes_mayores_40 = len(df[df['AGE'] > 40])
print(f"Total de pacientes con edad mayor a 40: {pacientes_mayores_40}")

pacientes_menores_40 = len(df[df['AGE'] < 40])
print(f"Total de pacientes con edad menor a 40: {pacientes_menores_40}")

Total Masculino 207
Total Femenino 235
-------------------------------------------
Total de pacientes con edad mayor a 40: 320
Total de pacientes con edad menor a 40: 117


Ambos casos cuentan con un tamaño de muestra mayor a 50, por lo tanto usamos Kolmogorov-Smirnov.

Hipótesis para Kolmogorov-Smirnoff
- $H_0$(Hipótesis Nula): Los datos sí siguen una distribución normal.
- $H_1$(Hipótesis Alterna): Los datos no siguen una distribución normal.

In [17]:
alpha = 0.05

datos_hombres_s1 = df[df['SEX'] == 2]['Y']
ks_stat, ks_p = stats.kstest(datos_hombres_s1, 'norm', args=(np.mean(datos_hombres_s1), np.std(datos_hombres_s1)))

print("Para hombres:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de Y en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de Y no siguen una DISTRIBUCIÓN NORMAL.")

#------------------------------------------------------------

datos_mujeres_s1 = df[df['SEX'] == 1]['S1']
ks_stat, ks_p = stats.kstest(datos_hombres_s1, 'norm', args=(np.mean(datos_mujeres_s1), np.std(datos_mujeres_s1)))

print("Para mujeres:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de Y en mujeres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de Y no siguen una DISTRIBUCIÓN NORMAL.")

print("-------------------------------------------------------------------------------------------------------------------------------")

pacientes_mayores_40 = df[df['AGE'] > 40]['Y']
ks_stat, ks_p = stats.kstest(pacientes_mayores_40, 'norm', args=(np.mean(pacientes_mayores_40), np.std(pacientes_mayores_40)))

print("Para Mayores de 40 años:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de Y en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de Y no siguen una DISTRIBUCIÓN NORMAL.")

# ------------------------------------

pacientes_menores_40 = df[df['AGE'] < 40]['Y']
ks_stat, ks_p = stats.kstest(pacientes_menores_40, 'norm', args=(np.mean(pacientes_menores_40), np.std(pacientes_menores_40)))

print("Para Menores de 40 años:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de Y en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de Y no siguen una DISTRIBUCIÓN NORMAL.")


Para hombres:
El p-valor (0.02423182669201318) es menor a 0.05. Asumimos que los datos de Y no siguen una DISTRIBUCIÓN NORMAL.
Para mujeres:
El p-valor (3.575320306388201e-35) es menor a 0.05. Asumimos que los datos de Y no siguen una DISTRIBUCIÓN NORMAL.
-------------------------------------------------------------------------------------------------------------------------------
Para Mayores de 40 años:
El p-valor (0.01001478728617788) es menor a 0.05. Asumimos que los datos de Y no siguen una DISTRIBUCIÓN NORMAL.
Para Menores de 40 años:
El p-valor (0.138900163709896) es mayor a 0.05. Asumimos que los datos de Y en hombres son NORMALES.


Como las muestras no tienen una distribución normal, pero son independiente, podemos usar una _**Prueba U de Mann-Whitney**_ para ambos casos.

Hipótesis para U de Mann-Whitney:
- $H_0$: Las medianas son iguales.
- $H_1$: Las medianas son diferentes.

In [23]:
datos1 = datos_hombres_s1 = df[df['SEX'] == 2]['Y']
datos2 = datos_mujeres_s1 = df[df['SEX'] == 1]['S1']
direction = "two-sided"
stat,p_value=stats.mannwhitneyu(datos1,datos2,alternative=direction)

print("1. La media de progresión de la enfermedad entre hombres y mujeres.")
if p_value < alpha:
    print(f"El p-valor ({p_value}) es menor a 0.05. Por lo tanto H0 es FALSO.")
else:
    print(f"El p-valor ({p_value}) es mayor a 0.05. Por lo tanto H0 es VERDADERO.")

print("-------------------------------------------------------------------------------------------------------------------------------")


datos1 = pacientes_mayores_40 = df[df['AGE'] > 40]['Y']
datos2 = pacientes_menores_40 = df[df['AGE'] < 40]['Y']
direction = "two-sided"
stat,p_value=stats.mannwhitneyu(datos1,datos2,alternative=direction)

print("2. La media de progresión de la enfermedad entre pacientes con menos de 40 años y más de 40 años de edad.")
if p_value < alpha:
    print(f"El p-valor ({p_value}) es menor a 0.05. Por lo tanto H0 es FALSO.")
else:
    print(f"El p-valor ({p_value}) es mayor a 0.05. Por lo tanto H0 es VERDADERO.")

1. La media de progresión de la enfermedad entre hombres y mujeres.
El p-valor (6.398239922452473e-08) es menor a 0.05. Por lo tanto H0 es FALSO.
-------------------------------------------------------------------------------------------------------------------------------
2. La media de progresión de la enfermedad entre pacientes con menos de 40 años y más de 40 años de edad.
El p-valor (0.004796137135198579) es menor a 0.05. Por lo tanto H0 es FALSO.


Así:
1. La media de progresión de la enfermedad entre hombres y mujeres es diferente.
2. La media de progresión de la enfermedad entre pacientes con menos de 40 años y más de 40 años de edad es diferente.

# Punto 4
Se considera que una persona no tiene un peso ideal, si su ibm está por 
encima de 24.9 ¿Hay diferencia en el nivel de glucosa en sangre entre las 
personas con peso ideal y las que no?

Variables:
- BMI $\rightarrow$ body mass index
- S6 $\rightarrow$ blood sugar level

Compararemos la media de S6 entre persona con BMI <= 24.9 y BMI > 24.9

Analicemos si estas muestras presentan una distribución normal o no.

In [24]:
peso_ideal = len(df[df["BMI"] <= 24.9])
print(f"Personas con peso ideal: {peso_ideal}")

peso_NOideal = len(df[df["BMI"] > 24.9])
print(f"Personas sin peso ideal: {peso_NOideal}")



Personas con peso ideal: 188
Personas sin peso ideal: 254


Muestras mayores a 50, usamos Kolmogorov-Smirnov.

Hipótesis:
- $H_0$(Hipótesis Nula): Los datos sí siguen una distribución normal.
- $H_1$(Hipótesis Alterna): Los datos no siguen una distribución normal.

In [26]:
peso_ideal = df[df["BMI"] <= 24.9]["S6"]
ks_stat, ks_p = stats.kstest(peso_ideal, 'norm', args=(np.mean(peso_ideal), np.std(peso_ideal)))

print("Personas con peso ideal:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S6 en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de S6 no siguen una DISTRIBUCIÓN NORMAL.")

#----------------------------------------

peso_NOideal = df[df["BMI"] > 24.9]["S6"]
ks_stat, ks_p = stats.kstest(peso_NOideal, 'norm', args=(np.mean(peso_NOideal), np.std(peso_NOideal)))

print("Personas sin peso ideal:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S6 en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de S6 no siguen una DISTRIBUCIÓN NORMAL.")



Personas con peso ideal:
El p-valor (0.4879038598517631) es mayor a 0.05. Asumimos que los datos de S6 en hombres son NORMALES.
Personas sin peso ideal:
El p-valor (0.1960437750388928) es mayor a 0.05. Asumimos que los datos de S6 en hombres son NORMALES.


Ahora, hacemos prueba de _**Homocedasticidad (Levene)**_, las hipotesis para esta prueba son:
- $H_0$: Los dos grupos de datos presentan varianza constante.
- $H_1$: Los dos grupos de datos no presentan varianza constante.

In [27]:
var1 = df[df["BMI"] <= 24.9]["S6"] # peso ideal
var2 = df[df["BMI"] > 24.9]["S6"]# peso no ideal
levene_stat, levene_p = stats.levene(var1, var2)

if ks_p > alpha:
    print(f"El p-valor ({levene_p}) es mayor a 0.05. Así que H0 es VERDADERO y los dos grupos de datos presentan varianza constante.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Así que H0 es FALSO y los dos grupos de datos no presentan varianza constante.")

El p-valor (0.543271396397639) es mayor a 0.05. Así que H0 es VERDADERO y los dos grupos de datos presentan varianza constante.
